In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("churn.db")

active_risk = pd.read_sql("""
    SELECT *
    FROM customer_risk_scores
    WHERE Churn = 'No'
    ORDER BY ChurnProbability DESC;
""", conn)

print(active_risk.shape)
active_risk.head(10)

(5174, 8)


,customerID,Contract,tenure,MonthlyCharges,Churn,rule_risk_score,ChurnProbability,RiskTier
0,5150-ITWWB,Month-to-month,3,94.85,No,8,0.933,Critical
1,2545-EBUPK,Month-to-month,2,84.05,No,8,0.914,Critical
2,3320-VEOYC,Month-to-month,14,95.60,No,6,0.913,Critical
3,8161-QYMTT,Month-to-month,7,94.10,No,8,0.908,Critical
4,6630-UJZMY,Month-to-month,4,83.25,No,8,0.906,Critical
5,7225-CBZPL,Month-to-month,17,94.80,No,6,0.905,Critical
6,6350-XFYGW,Month-to-month,4,94.75,No,8,0.905,Critical
7,4929-XIHVW,Month-to-month,2,95.50,No,8,0.903,Critical
8,5935-FCCNB,Month-to-month,17,94.20,No,6,0.903,Critical
9,4847-TAJYI,Month-to-month,6,89.35,No,8,0.902,Critical


In [6]:
active_risk["AnnualValueAtRisk"] = active_risk["MonthlyCharges"] * 12

def recommended_action(row):
    if row["RiskTier"] == "Critical":
        if row["Contract"] == "Month-to-month":
            return "Call this week: offer 1-year contract discount"
        return "Call this week: personal retention offer"
    elif row["RiskTier"] == "High":
        return "Email: targeted discount / highlight unused services"
    elif row["RiskTier"] == "Medium":
        return "Add to loyalty nurture campaign"
    return "No action needed"

active_risk["RecommendedAction"] = active_risk.apply(recommended_action, axis=1)
active_risk.head(10)

,customerID,Contract,tenure,MonthlyCharges,Churn,rule_risk_score,ChurnProbability,RiskTier,AnnualValueAtRisk,RecommendedAction
0,5150-ITWWB,Month-to-month,3,94.85,No,8,0.933,Critical,1138.2,Call this week: offer 1-year contract discount
1,2545-EBUPK,Month-to-month,2,84.05,No,8,0.914,Critical,1008.6,Call this week: offer 1-year contract discount
2,3320-VEOYC,Month-to-month,14,95.60,No,6,0.913,Critical,1147.2,Call this week: offer 1-year contract discount
3,8161-QYMTT,Month-to-month,7,94.10,No,8,0.908,Critical,1129.2,Call this week: offer 1-year contract discount
4,6630-UJZMY,Month-to-month,4,83.25,No,8,0.906,Critical,999.0,Call this week: offer 1-year contract discount
5,7225-CBZPL,Month-to-month,17,94.80,No,6,0.905,Critical,1137.6,Call this week: offer 1-year contract discount
6,6350-XFYGW,Month-to-month,4,94.75,No,8,0.905,Critical,1137.0,Call this week: offer 1-year contract discount
7,4929-XIHVW,Month-to-month,2,95.50,No,8,0.903,Critical,1146.0,Call this week: offer 1-year contract discount
8,5935-FCCNB,Month-to-month,17,94.20,No,6,0.903,Critical,1130.4,Call this week: offer 1-year contract discount
9,4847-TAJYI,Month-to-month,6,89.35,No,8,0.902,Critical,1072.2,Call this week: offer 1-year contract discount


In [7]:
tier_summary = active_risk.groupby("RiskTier").agg(
    Customers=("customerID", "count"),
    AvgChurnProbability=("ChurnProbability", "mean"),
    AnnualRevenueAtRisk=("AnnualValueAtRisk", "sum")
).round(2)

tier_order = ["Critical", "High", "Medium", "Low"]
tier_summary = tier_summary.reindex(tier_order)
print(tier_summary)

total_at_risk = active_risk[active_risk["RiskTier"].isin(["Critical","High"])]["AnnualValueAtRisk"].sum()
n_at_risk = active_risk[active_risk["RiskTier"].isin(["Critical","High"])].shape[0]
print(f"\n{n_at_risk} customers are High/Critical risk = ${total_at_risk:,.0f}/year at stake")

          Customers  AvgChurnProbability  AnnualRevenueAtRisk
RiskTier                                                     
Critical        672                 0.79             649636.2
High           1244                 0.54             982906.2
Medium          933                 0.30             694970.4
Low            2325                 0.07            1476316.2

1916 customers are High/Critical risk = $1,632,542/year at stake


In [8]:
active_risk.to_sql("retention_action_list", conn, if_exists="replace", index=False)
print("Saved table: retention_action_list")

Saved table: retention_action_list


In [9]:
active_risk.to_csv("retention_action_list.csv", index=False)
tier_summary.to_csv("tier_summary.csv")

print("Exported: retention_action_list.csv, tier_summary.csv")
conn.close()

Exported: retention_action_list.csv, tier_summary.csv
